# Experiment 004 — Learned Cell Graph

**Status: REGISTERED / UNRUN.** This is the single manual Kaggle entrypoint for Experiment 004. It runs exactly one development seed and one session, then appends verified evidence to a source-specific GitHub results branch.

Use the existing Kaggle notebook: [bingjiangli/minicells](https://www.kaggle.com/code/bingjiangli/minicells). Copy/import this repository notebook's cells into that notebook and enable Internet for the pinned Hugging Face data read. Do not create a new Kaggle notebook for each run. This workflow does not use Kaggle's notebook-push or Dataset-upload APIs.

The frozen scientific record remains in [experiment.yaml](../../experiments/004-cell-graph/experiment.yaml), [protocol.json](../../experiments/004-cell-graph/protocol.json), and [004-cell-graph.yaml](../../configs/004-cell-graph.yaml). This notebook is the integrated execution guide, not a replacement scientific authority.

## Research lineage and scope

- Experiment 001 is a single completed development comparison, with no confirmation claim.
- Experiment 002's observed seed 42 was not supported; development seeds 123 and 456 remain missing.
- Experiment 003 seed 42 remains DEVELOPMENT_INCOMPLETE. Its Cell is an FFN computation unit with shared outer causal Attention.
- Experiment 004 keeps that same Cell definition and reruns all five comparison arms. The 003 hard-cell arm is a registered reproduction inside 004, not a rewrite of the 003 record.
- The intervention changes both the learned transition prior and the backup-gradient estimator. A positive result would not isolate graph topology as the cause.
- This is development evidence only. It is not formal confirmation, a claim of semantic Cell specialization, or a HybridCLM regression.

## Frozen design at a glance

A 61M-parameter causal Transformer uses eight FFN Cells per layer, shared causal Attention, fixed entry Cell 0, at most four visits, and absorbing STOP. Each Cell retains its contextual transition head. Experiment 004 adds a learned 8×9 per-layer prior over eight destinations plus STOP. The model has five matched arms: fixed-cell, global-router, soft-cell, hard-cell, and graph-cell. Every arm targets 10M training tokens and uses the same pinned TinyStories revision and train-only 8,192-token BPE tokenizer.

| Session | GPU queue 0 | GPU queue 1 |
|---|---|---|
| reference | soft-cell → fixed-cell | global-router |
| graph | graph-cell | hard-cell |

The run requires two Tesla T4 GPUs. Workers are independent; there is no DDP. Manual execution has no preparation, session, training-arm or evaluation time cap. Each arm runs to its registered 10M token target, then finishes validation and evidence publication. The old budget fields are historical automatic-execution settings and do not stop manual workers. The authorized execution change is recorded in [manual-execution.md](../../experiments/004-cell-graph/manual-execution.md).

## Registered gates

All five full-budget arms, matched capacities, identical source/data/protocol hashes, and all six paired sessions must exist before the matrix can support any development conclusion.

For each seed, the graph arm must have lower mean full-path entropy than hard-cell, validation perplexity at most 13, measured forward GEMM FLOPs per token below half of fixed-cell, and at least two hard paths plus two used non-entry Cells in every layer. Lower entropy alone can mean collapse. Token/Cell mutual information is descriptive and finite-sample biased.

Timeouts, OOMs, missing measurements, or incomplete validation remain incomplete evidence. Never reduce token targets or overwrite an observed session. Preserve the execution policy and source identity of every observed run.

## Run matrix and source pin

Run one cell execution for each row in the matrix below. Select the seed and session in the next cell.

| Seed | Session 1 | Session 2 |
|---:|---|---|
| 42 | reference | graph |
| 43 | reference | graph |
| 44 | reference | graph |

For the first run, SOURCE_COMMIT may be main. Copy the exact source SHA printed by setup into SOURCE_COMMIT before running the paired session and all remaining rows. This pins all six sessions to one source revision. Default selection is seed 42 / reference.

The setup cell reads Kaggle secrets through Kaggle's in-notebook UserSecretsClient path. GITHUB_TOKEN is required for the preflight and append-only GitHub push. HF_TOKEN is required for the pinned Hugging Face input preparation and is removed before GPU workers start. The GPU cell runs only after the GitHub write preflight, two-T4 check, source/protocol checks, and pinned input preparation succeed.

After execution, the notebook commits evidence to results/cell-graph-SOURCE_SHA. If GitHub publication fails, rerun only the final publication cell; it will not rerun training.

## Frozen training and measurement details

The data identity is TinyStories revision f54c09fd23315a6f9c86f9dc80f725de7d8f9c64. The 8,192-token BPE tokenizer is trained only on the registered training stories.

Training follows the registered 25% soft phase at temperature 2, a 25% geometric anneal from 2 to 0.1 with soft forward values, and a 50% selected-hard phase. In the final phase the chosen Cell is the forward branch; the backward surrogate mixes the chosen and alternative top-2 branches evaluated at the same incoming state. Their probabilities are floored at 0.05 only for the backward mixture. STOP is identity. This is a biased surrogate; hard inference disables backup compute.

Validation covers 32,000 paired tokens. The recorded forward GEMM estimate includes Attention matmuls and excludes backward/non-matmul work. Preserve empirical edge counts separately from prior probabilities above 0.05, distinguish learned STOP from forced truncation, and retain per-layer path entropy, path diversity, non-entry Cell use, and descriptive token/Cell mutual information. Training-time visits and compute are not inferred from hard-inference FLOPs.

The full intervention is graph prior plus backup-gradient training. It does not isolate the causal effect of either component.

In [ ]:
from pathlib import Path
import json
import os
import re
import subprocess
import sys
import time

# Change only these three values between the six registered sessions.
SOURCE_COMMIT = "main"  # First session: main. Then pin its printed 40-character SHA.
SEED = 42               # Registered development seeds: 42, 43, 44.
SESSION = "reference"   # reference or graph.

KAGGLE_NOTEBOOK_URL = "https://www.kaggle.com/code/bingjiangli/minicells"
SETUP_STARTED = time.monotonic()
REPOSITORY_URL = "https://github.com/ArcheLabs/mini-cells.git"
ROOT = Path("/shared/kaggle/working/minicells-manual-004")

if not re.fullmatch(r"(main|[0-9a-f]{40})", SOURCE_COMMIT):
    raise ValueError("SOURCE_COMMIT must be main or a full lowercase Git SHA.")
if SEED not in (42, 43, 44) or SESSION not in ("reference", "graph"):
    raise ValueError("Choose a registered seed and session.")

In [ ]:
if ROOT.exists() and not (ROOT / ".git").exists():
    raise RuntimeError(f"Refusing to overwrite non-repository path: {ROOT}")
if not ROOT.exists():
    subprocess.run(
        ["git", "clone", "--depth=1", "--filter=blob:none", "--branch", "main",
         "--single-branch", REPOSITORY_URL, str(ROOT)],
        check=True
    )

# Reuse the repository Git helper for setup and publication.
sys.path.insert(0, str(ROOT / "scripts/research"))
from functools import partial
from publish_experiment_results import run_git as repository_run_git
run_setup_git = partial(repository_run_git, ROOT)

if SOURCE_COMMIT == "main":
    run_setup_git("fetch", "--depth=1", "origin", "main")
    SOURCE_SHA = run_setup_git("rev-parse", "FETCH_HEAD").stdout.strip()
else:
    run_setup_git("fetch", "--depth=1", "origin", SOURCE_COMMIT)
    SOURCE_SHA = SOURCE_COMMIT

RESULT_BRANCH = f"results/cell-graph-{SOURCE_SHA[:12]}"
ASSETS = Path(f"/shared/kaggle/working/minicells-004-assets-{SOURCE_SHA[:12]}")
PREPARATION = Path(f"/shared/kaggle/working/minicells-004-preparation-{SOURCE_SHA[:12]}-seed-{SEED}-{SESSION}.json")
remote = run_setup_git("ls-remote", "--exit-code", "--heads", "origin", RESULT_BRANCH, check=False)
if remote.returncode not in (0, 2):
    raise RuntimeError("Could not inspect the source-specific GitHub results branch.")
remote_exists = remote.returncode == 0

if remote_exists:
    run_setup_git("fetch", "--depth=32", "origin",
            f"refs/heads/{RESULT_BRANCH}:refs/remotes/origin/{RESULT_BRANCH}")
local_branch = run_setup_git("show-ref", "--verify", "--quiet",
                       f"refs/heads/{RESULT_BRANCH}", check=False).returncode == 0
current_branch = run_setup_git("rev-parse", "--abbrev-ref", "HEAD").stdout.strip()
dirty = run_setup_git("status", "--porcelain").stdout.strip()

if current_branch == RESULT_BRANCH:
    if dirty:
        print("Existing local files will be preserved; only registered result paths are staged below.")
    if remote_exists:
        remote_sha = run_setup_git("rev-parse", f"refs/remotes/origin/{RESULT_BRANCH}").stdout.strip()
        head_sha = run_setup_git("rev-parse", "HEAD").stdout.strip()
        if run_setup_git("merge-base", "--is-ancestor", remote_sha, head_sha, check=False).returncode:
            if run_setup_git("merge-base", "--is-ancestor", head_sha, remote_sha, check=False).returncode:
                raise RuntimeError("Local and GitHub result branches diverged; refusing to reset either.")
            if dirty:
                raise RuntimeError("GitHub advanced while local result files are uncommitted; preserve them and reconcile before continuing.")
            run_setup_git("merge", "--ff-only", f"origin/{RESULT_BRANCH}")
elif local_branch:
    raise RuntimeError("A local result branch already exists. Switch to it to preserve its commits before continuing.")
elif remote_exists:
    run_setup_git("checkout", "-b", RESULT_BRANCH, f"origin/{RESULT_BRANCH}")
else:
    run_setup_git("checkout", "-b", RESULT_BRANCH, SOURCE_SHA)

if run_setup_git("merge-base", "--is-ancestor", SOURCE_SHA, "HEAD", check=False).returncode:
    raise RuntimeError("The results branch does not descend from the pinned source commit.")
protected = [
    "src/minicells", "shared/python/experiments/common",
    "research/projects/cellular-language-models/experiments/004-cell-graph/config/registered.yaml",
    "research/projects/cellular-language-models/experiments/004-cell-graph/experiment.yaml",
    "research/projects/cellular-language-models/experiments/004-cell-graph/protocol.json",
]
if run_setup_git("diff", "--quiet", SOURCE_SHA, "HEAD", "--", *protected, check=False).returncode:
    raise RuntimeError("The results branch changed frozen implementation/config/protocol files.")
for comparison in (("diff", "--quiet", SOURCE_SHA), ("diff", "--cached", "--quiet", SOURCE_SHA)):
    if run_setup_git(*comparison, "--", *protected, check=False).returncode:
        raise RuntimeError("Local changes altered frozen implementation/config/protocol files.")

sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts/research"))
os.environ["PYTHONPATH"] = os.pathsep.join((str(ROOT / "src"), str(ROOT)))
for module, package in (("datasets", "datasets>=2.18,<5"),
                        ("tokenizers", "tokenizers>=0.15,<1"),
                        ("matplotlib", "matplotlib>=3.7,<4")):
    try:
        __import__(module)
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q",
             "--disable-pip-version-check", package],
            check=True
        )

from minicells.research.config import digest, load_definition
from minicells.research.assets import load_assets, prepare

DEFINITION = ROOT / "research/projects/cellular-language-models/experiments/004-cell-graph/experiment.yaml"
descriptor, CONFIG, PROTOCOL = load_definition(DEFINITION)
if descriptor["status"] != "registered_unrun":
    raise RuntimeError("Experiment 004's registered status changed; inspect its protocol before running.")
if SEED not in CONFIG["seeds"] or SESSION not in CONFIG["sessions"]:
    raise ValueError("Seed/session is outside the frozen Experiment 004 matrix.")

ARCHIVE_TARGET = (
    ROOT / "research/projects/cellular-language-models/experiments/004-cell-graph/artifacts" / SOURCE_SHA /
    f"seed-{SEED}" / SESSION
)
ALREADY_ARCHIVED = ARCHIVE_TARGET.exists()

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
try:
    github_token = secrets.get_secret("GITHUB_TOKEN").strip()
except Exception as exc:
    raise RuntimeError("Cannot read Kaggle Secret GITHUB_TOKEN in this original Kaggle notebook.") from exc
if not github_token:
    raise RuntimeError("Kaggle Secret GITHUB_TOKEN is empty.")
try:
    hf_token = secrets.get_secret("HF_TOKEN").strip()
except Exception as exc:
    raise RuntimeError("Cannot read Kaggle Secret HF_TOKEN in this original Kaggle notebook.") from exc
if not hf_token:
    raise RuntimeError("Kaggle Secret HF_TOKEN is empty.")

from publish_core_validation_007 import preflight_push, push_branch
os.environ["GITHUB_TOKEN"] = github_token
try:
    preflight_push(RESULT_BRANCH, "GITHUB_TOKEN")
finally:
    os.environ.pop("GITHUB_TOKEN", None)

if not ALREADY_ARCHIVED:
    import torch
    if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
        raise RuntimeError("Experiment 004 requires exactly two GPUs; no training was started.")
    GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(2)]
    if any("T4" not in name for name in GPU_NAMES):
        raise RuntimeError(f"Experiment 004 requires dual Tesla T4 GPUs; found {GPU_NAMES}.")

    if ASSETS.exists() and not (ASSETS / "assets-manifest.json").is_file():
        import shutil
        shutil.rmtree(ASSETS)
    if not (ASSETS / "assets-manifest.json").is_file():
        if hf_token:
            os.environ["HF_TOKEN"] = hf_token
        try:
            prepare(DEFINITION, ASSETS, timeout_seconds=None)
        finally:
            os.environ.pop("HF_TOKEN", None)
    ASSETS_SHA256 = digest(ASSETS / "assets-manifest.json")
    load_assets(ASSETS, CONFIG, ASSETS_SHA256)
    preparation = {
        "experiment": "004-cell-graph",
        "source_commit": SOURCE_SHA,
        "seed": SEED,
        "session": SESSION,
        "definition_sha256": digest(DEFINITION),
        "config_sha256": digest(ROOT / descriptor["config"]),
        "protocol_sha256": digest(PROTOCOL),
        "assets_manifest_sha256": ASSETS_SHA256,
        "scientific_decision": False,
    }
    PREPARATION.write_text(json.dumps(preparation, indent=2) + "\n", encoding="utf-8")
    RUN_OUTPUT = Path("/shared/kaggle/working/minicells-004-runs") / SOURCE_SHA / f"seed-{SEED}" / SESSION
    if RUN_OUTPUT.exists() and not (RUN_OUTPUT / "evidence-manifest.json").is_file():
        raise RuntimeError("A prior partial run directory exists. Preserve it; do not overwrite or rerun this session.")
    print({
        "source_commit": SOURCE_SHA, "result_branch": RESULT_BRANCH,
        "seed": SEED, "session": SESSION, "gpu_names": GPU_NAMES,
        "assets_manifest_sha256": ASSETS_SHA256,
        "setup_seconds": round(time.monotonic() - SETUP_STARTED, 1),
        "execution_policy": "manual-token-budget",
    })
else:
    print({"archive_already_exists": str(ARCHIVE_TARGET), "gpu_work": "skipped", "result_branch": RESULT_BRANCH})

# Keep both tokens out of the offline runtime and its GPU worker processes.
os.environ.pop("GITHUB_TOKEN", None)
os.environ.pop("HF_TOKEN", None)
del github_token, hf_token


## One registered run

Run the next cell once. The existing offline runtime handles the two independent T4 queues and writes its evidence manifest. This step runs no Kaggle API calls and does not publish until the evidence is archived and committed.

In [ ]:
if ALREADY_ARCHIVED:
    print("This seed/session is already archived. Training is skipped.")
elif (RUN_OUTPUT / "evidence-manifest.json").is_file():
    print("A completed local evidence bundle already exists. Training is skipped; continue to archive it.")
else:
    from minicells.runtime.research import run
    os.environ.pop("EXPERIMENT_STARTED_AT", None)
    decision = run(
        DEFINITION,
        ASSETS,
        ASSETS_SHA256,
        SEED,
        RUN_OUTPUT,
        SOURCE_SHA,
        session=SESSION,
        manual=True,
    )
    print(json.dumps(decision, indent=2))

## Verify, archive, and commit

The existing collector verifies every evidence checksum and identity, writes the immutable archive, records the manual Kaggle notebook provenance, updates the research registry, and derives the paired matrix summary. It never overwrites an observed session.

If collection printed an archive path and commit then failed, replace the old version of the next code cell with this corrected version and run it again, followed by the final push cell. It checks the archive on disk and verifies its original identity and checksums. Do not run setup or training again. The existing source SHA, seed and session remain unchanged.


In [ ]:
# Publication reuses the repository Git helper and has no setup deadline.
# This cell can be retried after collection, even if setup state is stale.
from functools import partial
from publish_experiment_results import run_git as repository_run_git, validate_origin
publish_git = partial(repository_run_git, ROOT)
validate_origin(ROOT)
if publish_git("rev-parse", "--abbrev-ref", "HEAD").stdout.strip() != RESULT_BRANCH:
    raise RuntimeError("Switch to the original results branch before committing evidence.")

if not ARCHIVE_TARGET.exists():
    if not (RUN_OUTPUT / "evidence-manifest.json").is_file():
        raise RuntimeError("No complete evidence manifest was written; preserve the Kaggle output for diagnosis.")
    subprocess.run(
        [
            sys.executable, "shared/kaggle/collect.py",
            "--download", str(RUN_OUTPUT),
            "--preparation", str(PREPARATION),
            "--kernel-ref", KAGGLE_NOTEBOOK_URL,
            "--collection", "manual-notebook",
            "--notebook-url", KAGGLE_NOTEBOOK_URL,
        ],
        cwd=ROOT,
        check=True,
    )
else:
    manifest_path = ARCHIVE_TARGET / "evidence-manifest.json"
    if not manifest_path.is_file():
        raise RuntimeError("An archive directory exists without its evidence manifest; preserve it for inspection.")
    archived_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    expected_identity = {
        "source_commit": SOURCE_SHA,
        "seed": SEED,
        "session": SESSION,
        "definition_sha256": digest(DEFINITION),
        "config_sha256": digest(ROOT / descriptor["config"]),
        "protocol_sha256": digest(PROTOCOL),
        "scientific_decision": False,
    }
    for key, value in expected_identity.items():
        if archived_manifest.get(key) != value:
            raise RuntimeError(f"Existing archive identity mismatch: {key}")
    required = {"metrics.json", "logs.jsonl", "routing_stats.json", "decision.json", "config.json", "experiment.json"}
    if not required.issubset(archived_manifest.get("files", {})):
        raise RuntimeError("Existing archive is missing required evidence files.")
    for name, checksum in archived_manifest["files"].items():
        file = (ARCHIVE_TARGET / name).resolve()
        if not file.is_relative_to(ARCHIVE_TARGET.resolve()) or not file.is_file() or digest(file) != checksum:
            raise RuntimeError(f"Existing archive checksum mismatch: {name}")
    print("The existing immutable archive passed identity and checksum verification.")

archive_rel = ARCHIVE_TARGET.relative_to(ROOT).as_posix()
matrix_rel = (ARCHIVE_TARGET.parent.parent / "matrix-analysis").relative_to(ROOT).as_posix()
publish_git("config", "user.name", "MiniCells Kaggle")
publish_git("config", "user.email", "kaggle@minicells.local")
allowed = (archive_rel, matrix_rel, "research/governance/registry.yaml")
staged = publish_git("diff", "--cached", "--name-only", "-z").stdout.split("\0")
if any(path and not any(path == item or path.startswith(item + "/") for item in allowed)
       for path in staged):
    raise RuntimeError("Unrelated files are staged; preserve them before publishing this session.")
publish_git("add", "--", *allowed)
if publish_git("diff", "--cached", "--quiet", check=False).returncode:
    message = f"research: archive Experiment 004 seed {SEED} {SESSION}"
    publish_git("commit", "-m", message)
    print({"commit": publish_git("rev-parse", "HEAD").stdout.strip(), "branch": RESULT_BRANCH})
else:
    print({"commit": "no new files", "branch": RESULT_BRANCH})

## Push results to GitHub

This pushes only the source-specific results branch, using the repository's existing Kaggle Secret authentication helper and a normal non-force push. If this cell fails because of a temporary GitHub or token issue, correct the secret and rerun this cell; do not rerun training. A later seed/session uses the same source SHA and branch.

For an already-open session using the older notebook, run this small compatibility repair in a new cell, then rerun only the old archive/commit cell and the final push cell:

```python
from functools import partial
from publish_experiment_results import run_git as existing_git
run_git = partial(existing_git, ROOT)
ALREADY_ARCHIVED = ARCHIVE_TARGET.exists()
```

This uses the existing Git helper and preserves the archived source/protocol hashes. If the Kaggle kernel has already stopped, preserve/download its working output before recovery; this cell requires the original checkout and archive to remain available.


In [ ]:
# The publisher reads the current Kaggle Secret and scopes it to Git's askpass process.
push_branch(RESULT_BRANCH, "GITHUB_TOKEN")
